# Week 3 — 적대적 학습: pix2pix GAN

## 이번 주 학습 목표
1. **L1 손실이 왜 흐린 결과를 내는지** — "평균의 함정"을 연속 출력으로 직접 확인
2. **판별자(Discriminator)** 를 만들고, **hinge 손실**을 숫자로 이해
3. **경로① 처음부터** GAN 학습 · **경로② W2 체크포인트 이어받아** 미세조정 — 둘 다 실습
4. GAN 없음 vs GAN 을 **연속 출력·경계·지표**로 비교
5. (심화) **λ sweep** 으로 선명도 ↔ |Δφ| trade-off, 그리고 *"좋은 숫자 ≠ 좋은 복원"* 을 정직하게 평가

> 이번 주는 이웃이 먼 어려운 보간 **k=5** 로 실습합니다. k=1(가까운 이웃)에서는 L1도 이미 거의 완벽해 GAN의 효과가 잘 안 보이기 때문입니다.

## 0. 환경 준비

In [ ]:
import sys
from pathlib import Path

# helpers(dr_utils.py · model_utils.py)는 같은 폴더(다운로드 zip) 또는 ../helpers(저장소)에 있을 수 있음
for _cand in [Path('.'), Path('..') / 'helpers']:
    if (_cand / 'dr_utils.py').exists():
        sys.path.insert(0, str(_cand.resolve())); break

import numpy as np
import matplotlib.pyplot as plt
import torch

from dr_utils import (
    load_volume, porosity, predict_linear_k, eval_targets,
    setup_plot_style, ORANGE, NAVY, GREEN, RED, GRAY,
)
from model_utils import (
    UNetMini, count_parameters, train_quick, evaluate_model, load_ckpt,
    # --- W3 신규 ---
    PatchDiscriminatorMini, ssim_loss, d_hinge_loss, g_hinge_loss,
    train_gan, predict_continuous, save_gan_ckpt, load_gan_ckpt, GAN_PRESETS,
)
setup_plot_style()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
print(f'PyTorch {torch.__version__}, device={DEVICE}')

## 1. W2 복습 — 그런데 왜 아직 뿌옇지?

W2의 UNet은 선형 보간을 크게 이겼습니다. 하지만 **threshold(0.5) 하기 전의 연속 출력**을 보면,
L1만 쓴 모델은 pore 경계에서 **회색(불확실)** 으로 번집니다. 왜일까요?

> **평균의 함정**: 이웃 두 장만으로는 가운데 pore 위치가 애매합니다. L1은 오차를 줄이려 여러 후보의
> **평균(회색)** 을 출력합니다 — 어느 쪽도 아닌 흐릿한 값.

In [ ]:
DATA = next((p for p in [Path('data'), Path('..') / 'data'] if (p / 'BB_256.bin').exists()), Path('data'))
bb = load_volume(DATA / 'BB_256.bin')
K = 5   # 이웃 거리 (어려운 보간)

# 선형 baseline (이길 대상)
m_lin = eval_targets(predict_linear_k(bb, K), bb, K)
print(f'선형 보간 (k={K})   |Δφ|={m_lin["dphi_pp"]:.2f}%p   SSIM={m_lin["ssim"]:.4f}')

# L1만 쓴 mini 모델 (빠르게) — GAN 없음 상태
#   train_gan(lambda_gan=0, w_ssim=0) = 순수 L1 학습기
print('\nL1만 학습 (GAN 없음)…')
G_l1, _, _ = train_gan(bb, k=K, preset='fast', lambda_gan=0.0, w_ssim=0.0,
                       epochs=18, warmup=0, device=DEVICE, verbose=True)

### 흐림을 눈으로 — 연속 출력과 가로 단면

In [ ]:
# 고정 관찰 patch (실제 triplet)
z, c0, c1 = 128, 64, 192
before, after, target = bb[z-K, c0:c1, c0:c1], bb[z+K, c0:c1, c0:c1], bb[z, c0:c1, c0:c1]

cont_l1 = predict_continuous(G_l1, before, after, device=DEVICE)
grey = lambda c: ((c > 0.2) & (c < 0.8)).mean()   # 회색(불확실) 비율

fig, ax = plt.subplots(1, 3, figsize=(13, 4.4))
ax[0].imshow(target, cmap='gray_r', vmin=0, vmax=1); ax[0].set_title('원본 (정답)')
ax[1].imshow(cont_l1, cmap='gray_r', vmin=0, vmax=1); ax[1].set_title(f'L1 연속 출력 · 회색 {grey(cont_l1)*100:.0f}%')
row = int(np.argmax(np.abs(np.diff(target, axis=1)).sum(axis=1)))
for a in ax[:2]: a.axhline(row, color=ORANGE, ls='--', lw=1.3); a.set_xticks([]); a.set_yticks([])
ax[2].plot(target[row], color=NAVY, lw=2.4, label='정답 (계단)')
ax[2].plot(cont_l1[row], color=RED, lw=2.2, label='L1 (완만 = 회색)')
ax[2].axhline(0.5, color=GRAY, ls=':', lw=1); ax[2].legend(fontsize=9); ax[2].set_title('가로 단면')
plt.tight_layout(); plt.show()
print('L1은 경계에서 0.5 근처 회색으로 번집니다 → 진짜 암석엔 없는 값.')

## 2. 판별자 (Discriminator) 만들기

GAN의 핵심은 두 번째 신경망 **판별자 D** 입니다. 슬라이스를 받아 *"진짜 암석 단면인가?"* 를 판정합니다.
우리 D는 세 가지 장치를 씁니다:

- **조건부(conditional)**: 이웃 슬라이스 `[t−k, t+k]` 를 함께 입력 → "사실적이면서 **이웃과 일관**된가"
- **PatchGAN**: 이미지를 겹치는 **조각(patch)** 으로 나눠 각각 진짜/가짜 점수 (국소 텍스처·경계에 집중)
- **spectral norm**: 판별자의 힘에 상한 → 학습 안정화

In [ ]:
D = PatchDiscriminatorMini(cond_ch=2, base=32)
print('D 파라미터:', f'{count_parameters(D):,}')

# 입력: 조건(2ch) + 판정 대상(1ch) → patch별 점수 map
cond = torch.randn(1, 2, 64, 64)
y    = torch.rand(1, 1, 64, 64)
score = D(cond, y)
print('입력 조건', tuple(cond.shape), '+ 대상', tuple(y.shape), '→ 점수 map', tuple(score.shape))
print('점수 map의 각 칸 = 한 patch의 진짜(+)/가짜(−) 정도')

## 3. 적대적 손실 — hinge

**판별자 D** 는 진짜를 +1 이상, 가짜를 −1 이하로 밀어냅니다:

$$\mathcal{L}_D = \mathbb{E}\,[\max(0,\,1-D(y))] + \mathbb{E}\,[\max(0,\,1+D(G(x)))]$$

**생성자 G** 는 판별자를 속이려(점수를 높이려) 합니다:

$$\mathcal{L}_G^{\text{adv}} = -\,\mathbb{E}\,[D(G(x))]$$

"확실히 맞히면 벌점 0, 애매하면 벌점" — 숫자로 확인해 봅시다.

In [ ]:
real_score = torch.tensor([0.8, 1.5, -0.2])   # 진짜에 매긴 점수
fake_score = torch.tensor([-0.4, 0.3, -1.5])  # 가짜에 매긴 점수
d_pen = torch.relu(1 - real_score).mean() + torch.relu(1 + fake_score).mean()
g_adv = -fake_score.mean()
print('D 벌점 (진짜는 ≥+1, 가짜는 ≤−1 이면 0):', round(float(d_pen), 3))
print('G 적대적 손실 (−D(fake), 작을수록 잘 속임):', round(float(g_adv), 3))
print('→ real=1.5, fake=−1.5 는 벌점 0 (확실히 맞힘). real=0.8, fake=0.3 은 벌점 발생.')

## 4. 경로 ① — 처음부터 GAN 학습

`train_gan` 이 warmup·hinge·spectral norm 을 모두 처리합니다.
처음 몇 epoch 은 **재구성(L1·SSIM)만** 으로 몸을 풀고(λ=0), 이후 판별자를 붙입니다.
`snapshot` 을 주면 학습 중 연속 출력을 저장해 progression 을 볼 수 있습니다.

In [ ]:
G, D, hist = train_gan(bb, k=K, preset='fast', lambda_gan=0.12, w_ssim=0.3,
                       lambda_decay=0.3, epochs=30, warmup=8,
                       snapshot=(before, after), snapshot_every=3,
                       device=DEVICE, verbose=True)
save_gan_ckpt(G, D, 'w3_gan_mini.pth', meta={'g_base': 8, 'd_base': 32, 'k': K})
print('\n체크포인트 저장: w3_gan_mini.pth')

### 학습 곡선 — G와 D의 줄다리기

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12.5, 4.2))
ax[0].plot(hist['G_l1'], color=NAVY, label='G · L1')
ax[0].plot(hist['G_ssim'], color=ORANGE, label='G · SSIM loss')
ax[0].axvline(8, color=GRAY, ls='--'); ax[0].text(8.3, max(hist['G_l1'])*0.9, 'GAN 켜짐', color=GRAY)
ax[0].set_title('생성자 재구성 손실'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(hist['D_loss'], color=RED, label='D 손실')
ax[1].plot(hist['G_gan'], color=GREEN, label='G 적대적')
ax[1].set_title('적대적 손실 (D vs G)'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()
print('D·G 손실을 함께 보세요. 한쪽이 0에 붙거나 크게 진동하면 불안정 신호입니다.')

### GAN 출력은 선명해졌나 — L1과 나란히

In [ ]:
cont_gan = predict_continuous(G, before, after, device=DEVICE)
fig, ax = plt.subplots(1, 3, figsize=(13, 4.4))
for a, im, t, col in [(ax[0], target, '원본', NAVY), (ax[1], cont_l1, f'L1 · 회색 {grey(cont_l1)*100:.0f}%', RED),
                      (ax[2], cont_gan, f'GAN · 회색 {grey(cont_gan)*100:.0f}%', GREEN)]:
    a.imshow(im, cmap='gray_r', vmin=0, vmax=1); a.set_title(t, color=col); a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print(f'회색(불확실) 비율 — L1 {grey(cont_l1)*100:.0f}% → GAN {grey(cont_gan)*100:.0f}%  (낮을수록 선명)')

## 5. 경로 ② — W2 체크포인트 이어받아 미세조정

처음부터가 아니라, **W2에서 학습한 UNet** 을 Generator 초기값으로 이어받아 GAN 으로 미세조정할 수도 있습니다.
(여기서는 W2 모델을 빠르게 하나 만들어 시연 — 실제로는 W2에서 저장한 `unet_mini_*.pth` 를 `load_ckpt` 로 불러오면 됩니다.)

In [ ]:
# W2 스타일 UNet 하나 빠르게 준비 (실제로는: G0, _ = load_ckpt('unet_mini_fast.pth'))
G0, _ = train_quick(bb, k=K, preset='fast', device=DEVICE, verbose=False)
print('W2 UNet 준비 완료. 이어서 GAN 미세조정…')

G_ft, D_ft, hist_ft = train_gan(bb, k=K, generator=G0, lambda_gan=0.12, w_ssim=0.3,
                                epochs=15, warmup=2, device=DEVICE, verbose=True)
cont_ft = predict_continuous(G_ft, before, after, device=DEVICE)
print(f'\n미세조정 결과 회색 비율: {grey(cont_ft)*100:.0f}%  (W2 이어받기는 warmup을 짧게 줄일 수 있음)')

## 6. 평가 — GAN 없음 vs GAN (정직하게)

여기서 **이번 주 가장 중요한 교훈** 이 나옵니다. GAN을 켜면 |Δφ|·SSIM 이 *더 좋아질까요?*
직접 재보고, 결과를 정직하게 해석합니다.

In [ ]:
res_l1  = evaluate_model(G_l1, bb, k=K, device=DEVICE)
res_gan = evaluate_model(G,    bb, k=K, device=DEVICE)
print(f'{"방법":<14}{"|Δφ|(%p)":>10}{"SSIM":>9}')
print(f'{"선형":<14}{m_lin["dphi_pp"]:>10.2f}{m_lin["ssim"]:>9.3f}')
print(f'{"UNet · L1":<14}{res_l1["dphi_pp"]:>10.2f}{res_l1["ssim"]:>9.3f}')
print(f'{"UNet + GAN":<14}{res_gan["dphi_pp"]:>10.2f}{res_gan["ssim"]:>9.3f}')

> **관찰**: GAN이 흐림(회색 경계)을 없애 pore 구조를 사실적으로 복원하면서
> **|Δφ|·SSIM까지 개선**됩니다 — 회색이 0.5 근처에서 threshold될 때 생기던 오차가 사라지기 때문이죠.
> 이는 우리 **연구의 큰 모델**에서도 같습니다: 볼륨 전체 |Δφ|가 GAN 없음 → GAN에서 낮아지고
> (BB 예: 0.49 → 0.19 %p), **투과율·S2 같은 물리 물성도 함께 개선**됩니다.
>
> **그래도 |Δφ| 하나로는 부족합니다**: 같은 |Δφ|라도 pore **연결성**이 다르면 투과율이 달라집니다.
> |Δφ|는 부피 비율일 뿐이라 구조를 다 담지 못하죠. 그래서 flow가 걸린 문제는 **S2 상관·투과율**(W5)
> 같은 **물리 지표**로 평가합니다 — GAN이 개선하는 것이 바로 이 구조입니다.
> **단, 단일 슬라이스 |Δφ| 하나로 판단하지 말 것.**

In [ ]:
# 이진 복원을 시각으로 — 경계·작은 pore 차이 확인
zc = K  # 예측 슬라이스 하나
b_l1  = (res_l1['recon'][z] > 0.5) if 'recon' in res_l1 else None
fig, ax = plt.subplots(1, 3, figsize=(13, 4.4))
ax[0].imshow(bb[z], cmap='gray_r', vmin=0, vmax=1); ax[0].set_title('원본')
ax[1].imshow(res_l1['recon'][z], cmap='gray_r', vmin=0, vmax=1); ax[1].set_title('UNet · L1')
ax[2].imshow(res_gan['recon'][z], cmap='gray_r', vmin=0, vmax=1); ax[2].set_title('UNet + GAN')
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 6.5 심화 — λ sweep: 흐림 ↔ 환각의 균형

adversarial 가중치 **λ** 하나가 균형을 정합니다. 너무 작으면 L1과 같아 흐리고, 너무 크면 없는 구조를 만드는 **환각** 위험.
직접 몇 값을 바꿔 **선명도(회색 비율↓)** 와 **|Δφ|** 를 관찰합니다. (빠르게 하려고 epoch 을 작게 둡니다.)

In [ ]:
rows = []
for lam in [0.0, 0.1, 0.5]:
    Gi, _, _ = train_gan(bb, k=K, preset='fast', lambda_gan=lam, w_ssim=0.3,
                         epochs=14, warmup=4, device=DEVICE, verbose=False)
    ci = predict_continuous(Gi, before, after, device=DEVICE)
    ri = evaluate_model(Gi, bb, k=K, device=DEVICE)
    rows.append((lam, grey(ci)*100, ri['dphi_pp'], ri['ssim']))
    print(f'λ={lam:<4}  회색 {grey(ci)*100:4.0f}%   |Δφ| {ri["dphi_pp"]:.2f}%p   SSIM {ri["ssim"]:.3f}')
print('\n작은 λ = 흐림(회색↑), 큰 λ = 선명하지만 |Δφ|가 흔들릴 수 있음 → 적정 λ를 찾는 게 핵심.')

## 7. 정리 & 다음 주

**오늘 배운 것**
- L1은 "평균의 함정"으로 경계를 회색으로 흐린다 (연속 출력으로 확인)
- GAN = 생성자 ↔ 판별자의 경쟁 · 조건부 · PatchGAN · spectral norm · hinge 손실
- 두 경로(처음부터 / W2 이어받기) 모두로 GAN 학습
- **GAN은 구조를 사실적으로 복원** → |Δφ|(볼륨)와 투과율·S2 같은 물성을 함께 개선한다. 단 단일 슬라이스 숫자 하나로 판단하지 말 것 (→W5 물리 지표).

**다음 주 (W4)** — 다른 아키텍처 (Transformer 기반 · 3D). GAN 체크포인트(`w3_gan_mini.pth`)를 보존하세요.

**탐구 과제** (핸드아웃 §5): GAN 있음/없음 경계 비교 · λ sweep trade-off · warmup·spectral norm 영향 · "숫자는 비슷한데 구조는 다르다" 본인 예시.